## Mundakkai Hill Subsidence ( Wayanad - 2024 )

The InSAR.dev ecosystem, the PyGMTSAR InSAR library, the Geomed3D geophysical inversion library, and N-Cube 3D/4D GIS data visualization (among others) are open-source projects I develop in my free time.

I hold a Master’s degree in STEM, specializing in radio physics. In 2004, I received first prize in the All-Russian Physics Competition for significant results in forward and inverse modeling for nonlinear optics and holography. These skills are also applicable to modeling gravity, magnetic, and thermal fields, as well as satellite interferometry processing.

With 20 years of experience as a data scientist and software developer, I have contributed to scientific and industrial development through government contracts, university projects, and work with companies including LG Corp and Google Inc.

You can support my work on Patreon, where I share updates on my projects, publications, use cases, examples, and other useful information. For research and development services and support, please visit my profile on the freelance platform Upwork.

You can support my work on [Patreon](https://www.patreon.com/pechnikov), where I share updates on my projects, publications, use cases, examples, and other useful information. For research and development services and support, please visit my profile on the freelance platform [Upwork](https://www.upwork.com) or reach out to me directly.

### Resources
- Google Colab Pro notebooks and articles on [Patreon](https://www.patreon.com/pechnikov),
- Google Colab notebooks on [GitHub](https://github.com),
- Docker Images on [DockerHub](https://hub.docker.com),
- Geological Models on [YouTube](https://www.youtube.com),
- VR/AR Geological Models on [GitHub](https://github.com),
- Live updates and announcements on [LinkedIn](https://www.linkedin.com/in/alexey-pechnikov/).

© Alexey Pechnikov, 2025

$\large\color{blue}{\text{Hint: Use menu Cell} \to \text{Run All or Runtime} \to \text{Complete All or Runtime} \to \text{Run All}}$
$\large\color{blue}{\text{(depending of your localization settings) to execute the entire notebook}}$

# Stage 1. InSAR.dev-PyGMTSAR: A Python package for InSAR pre-processing

Convert Sentinel-1 SLC data into a geocoded, cloud-ready Zarr dataset. The output can be hosted on GitHub or any file storage as a set of files or a single ZIP archive.

## Google Colab Installation

Install InSAR.dev Python libraries

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !{sys.executable} -m pip install -q pyvista xvfbwrapper jupyter_bokeh vtk panel
    #trame trame-vuetify trame-vtk
    from google.colab import output
    output.enable_custom_widget_manager()
    # install the exact commit (no pip cache)
    !{sys.executable} -m pip install --no-cache-dir \
      "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev_toolkit"
    !{sys.executable} -m pip install --no-cache-dir \
      "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev_pygmtsar"
    !{sys.executable} -m pip install --no-cache-dir \
      "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev"

## Import Python Modules

In [ ]:
import numpy as np
import geopandas as gpd
import shapely
import json
import matplotlib.pyplot as plt
%matplotlib inline
# setup dark theme
from insardev_toolkit.UI import UI
UI('dark')

In [ ]:
# print versions
from insardev_pygmtsar import __version__ as pygmtsar_version
from insardev_toolkit import __version__ as toolkit_version
print("insardev_pygmtsar version:", pygmtsar_version)
print("insardev_toolkit version:", toolkit_version)
# import modules to be used in the notebook
from insardev_pygmtsar import S1
from insardev_toolkit import EOF, ASF, Tiles, XYZTiles

## Specify Sentinel-1 SLC Bursts and Area

Note :  The Sentinel Burst data is already programmatically acquired from asf API and stored in 'output_asf'. Code for the data acquisition can be found in ' stac_asf.py '. 

## Specify Directories

In [ ]:
# directory where Sentinel-1 bursts and orbits will be downloaded
DATADIR = 'output_asf_pre'
# path to the downloaded DEM file
DEM = f'{DATADIR}/dem.nc'
ZARRDIR = 'zarr'

In [ ]:
# read the notices printed below carefully
s1 = S1(DATADIR)
s1.to_dataframe().to_file(f"{DATADIR}/s1.geojson", driver="GeoJSON")
#df = gpd.read_file("s1.geojson")
s1.to_dataframe()

In [ ]:
# scan the data directory for Sentinel-1 SLC bursts and download related orbits
EOF().download(DATADIR, s1.to_dataframe())

In [ ]:
s1.to_dataframe()

In [ ]:
# download Copernicus Global DEM 1 arc-second
Tiles().download_dem(s1.to_dataframe(), provider='GLO', filename=DEM)[::4,::4].plot.imshow(cmap='cividis')

## Preprocess Sentinel-1 SLC to Geocoded Zarr

In [ ]:
# scan SLC bursts with DEM
s1 = S1(DATADIR, DEM=DEM)
s1.to_dataframe()

In [ ]:
# descending orbit bursts: preview
s1.plot(ref='2024-08-01')

In [ ]:
BBOX = [76.12683, 11.457, 76.14517, 11.475]
s1.transform(ZARRDIR, ref='2024-08-01', bbox=BBOX)


# Stage 2. InSAR.dev: A Python package for InSAR processing

Load Sentinel-1 geocoded, cloud-ready Zarr dataset locally or hosted on GitHub or any file storage as a set of files or a single ZIP archive.

## Import Python Modules

In [ ]:
import matplotlib.pyplot as plt
%matplotlib inline
# setup dark theme
from insardev_toolkit.UI import UI
UI('dark')

In [ ]:
import sys
if 'google.colab' in sys.modules:
    !{sys.executable} -m pip install -q --no-cache-dir \
        "git+https://github.com/AlexeyPechnikov/InSARdev.git#subdirectory=insardev"

In [ ]:
# print versions
from insardev import __version__ as insardev_version
from insardev_toolkit import __version__ as toolkit_version
print("insardev version:", insardev_version)
print("insardev_toolkit version:", toolkit_version)
# import modules to be used in the notebook
from insardev import Stack, BatchUnit
# data downloader
from insardev_toolkit.HTTP import unzip

In [ ]:
# common data science libraries
import xarray as xr
import numpy as np
import rioxarray as rio

In [ ]:
# 3D plotting modules
import pyvista as pv
# white background
#pv.set_plot_theme("document")
pv.set_plot_theme("dark")
from IPython.display import display, HTML
if 'google.colab' in sys.modules:
    import panel
    panel.extension(comms='ipywidgets')
    panel.extension('vtk')
# 2D interactive map modules
import json
import matplotlib.colors
from ipyleaflet import Map, GeoJSON, LayersControl, basemaps, basemap_to_tiles

## Specify Directories

In [ ]:
ZARRDIR = 'zarr'
WORKDIR = 'workdir'

## Download and Unpack Datasets

We can process datasets hosted on GitHub, Zenodo, or other platforms in a single workflow.

In [ ]:
# example downloading command
#unzip("https://zenodo.org/records/15347694/files/Türkiye_Elevation-40x10-004.zip", ZARRDIR)

## Run Local Dask Cluster

Launch a Dask cluster for local or distributed multicore computing. This makes it possible to process terabyte-scale Sentinel-1 SLC datasets even on an Apple MacBook Air with 16 GB of RAM.

In [ ]:
# simple Dask initialization
if 'client' in globals():
    client.shutdown()
    client.close()
# For stubborn memory, reset CUDA device
import torch
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.reset_peak_memory_stats()
if torch.backends.mps.is_available():
    # Don't call synchronize() - it can hang if there was an error!
    # Just empty cache and collect garbage
    torch.mps.empty_cache()
    gc.collect()
    # twice helps sometimes
    torch.mps.empty_cache()
import logging
logging.getLogger('distributed').setLevel(logging.CRITICAL)
from dask.distributed import Client
# set threads_per_worker=1 to prevent the error 'CUDA out of memory' on GPU instances
client = Client(silence_logs='CRITICAL', threads_per_worker=1, resources={'gpu': 1})
client

## Load Geocoded Bursts

In [ ]:
stack = Stack().load(ZARRDIR).align_elevation()
stack

In [ ]:
stack.plot(cmap='autumn', alpha=0.15)
# download the basemap adding the buffer to cover the area of interest after reprojecting
gmap = XYZTiles().download_googlesatellite(stack.to_dataframe().buffer(5_000), zoom=10, fill_value=0)
gmap.plot.imshow(ax=plt.gca(), zorder=-1, add_labels=False)
plt.gca().set_title(f'Sentinel-1 Burst Footprints');

## Compute Baseline

In [ ]:
baseline = stack.baseline(days=100, meters=150)
baseline

In [ ]:
baseline.plot()

In [ ]:
baseline.hist()

## Compute Interferograms

In [ ]:
# 100m antialiasing Gaussian filter and multilooking (spatial averaging)
intf, corr = stack.pairs(baseline.tolist()).interferogram(wavelength=100).goldstein(32).angle().downsample(2).compute()
intf


In [ ]:
# align and unify bursts
intf = intf.align().dissolve().compute()
corr = corr.dissolve().compute()
print(intf)

In [ ]:
intf.plot(rows=3, cols=3, size=2.5)


In [ ]:
corr.plot(rows=3, cols=3, size=2.5)

In [ ]:
stack.to_vtk('intf', intf.downsample(100))

In [ ]:
# build interactive 3D plot
plotter = pv.Plotter(notebook=True)
mesh = pv.read('intf/VV.vtk').scale([1, 1, 4], inplace=True)
# pick the first interferogram pair scalar from the VTK mesh
import re as _re
_pair_scalars = [n for n in mesh.array_names if _re.fullmatch(r'\d{8}_\d{8}', n)]
_scalar_name = _pair_scalars[0] if _pair_scalars else mesh.array_names[-1]
plotter.add_mesh(mesh, scalars=_scalar_name, cmap='turbo', ambient=0.1, show_scalar_bar=True)
plotter.show_axes()
bounds = mesh.bounds
center = mesh.center
# move camera closer by reducing distance (zoom in)
distance = mesh.length / 1.4
plotter.camera.position = (center[0] + distance, center[1] + distance, center[2] + distance * 0.5)
plotter.camera.focal_point = center
plotter.camera.azimuth = 215
plotter.camera.elevation = 15
plot = plotter.show(screenshot='3D Interferogram.png', jupyter_backend='panel' if 'google.colab' in sys.modules else 'client', return_viewer=True)
if 'google.colab' in sys.modules:
    plot = panel.panel(
        plotter.render_window, orientation_widget=plotter.renderer.axes_enabled,
        enable_keybindings=False, sizing_mode='stretch_width', min_height=600
    )
display(HTML('<h1 style="text-align:center; margin-bottom:0;">Interactive Sentinel-1 Interferogram on DEM</h1>'))
plot

## Unwrap Interferograms

In [ ]:
# unwrap as a single raster
phase2d = stack.unwrap2d_dataset(intf.to_dataset(), corr.to_dataset()).compute()
phase2d

In [ ]:
# convert back to burst stack and materialize
phase = intf.from_dataset(phase2d)

In [ ]:
# Convert phase BatchWrap to standard DataArray
sub_phase = phase.to_dataset().VV.compute()

# We want 3x3 subplots for the first 9 pairs
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
vmin = sub_phase.quantile(0.01).item()
vmax = sub_phase.quantile(0.99).item()

for i, ax in enumerate(axes.flat):
    if i >= 9 or i >= sub_phase.sizes["pair"]:
        ax.axis("off")
        continue
    d = sub_phase.isel(pair=i)
    im = ax.imshow(d.values, cmap='turbo', vmin=vmin, vmax=vmax)
    
    ref_date = str(d.ref.values)[:10]
    rep_date = str(d.rep.values)[:10]
    ax.set_title(f"{ref_date} to {rep_date}", fontsize=8)
    ax.axis("off")

fig.colorbar(im, ax=axes, shrink=0.6, label='Phase [rad]')
plt.show()


## Detrend Unwrapped Phase

In [ ]:
phase_detrend = phase - phase.gaussian(wavelength=1500)


In [ ]:
# Convert phase_detrend Batch/BatchWrap to standard DataArray
sub_detrend = phase_detrend.to_dataset().VV.compute()

# We want 3x3 subplots for the first 9 pairs
fig, axes = plt.subplots(3, 3, figsize=(12, 10))
vmin = sub_detrend.quantile(0.01).item()
vmax = sub_detrend.quantile(0.99).item()

for i, ax in enumerate(axes.flat):
    if i >= 9 or i >= sub_detrend.sizes["pair"]:
        ax.axis("off")
        continue
    d = sub_detrend.isel(pair=i)
    im = ax.imshow(d.values, cmap='turbo', vmin=vmin, vmax=vmax)
    
    ref_date = str(d.ref.values)[:10]
    rep_date = str(d.rep.values)[:10]
    ax.set_title(f"{ref_date} to {rep_date}", fontsize=8)
    ax.axis("off")

fig.colorbar(im, ax=axes, shrink=0.6, label='Detrended Phase [rad]')
plt.show()


## Convert Phase to Line-of-Sight (LOS) Displacement

In [ ]:
displacement = stack.displacement_los(phase_detrend)
displacement

## Compute Accumulated Displacement Time Series

In [ ]:
displacement_cum = stack.lstsq(displacement, corr).compute()
displacement_cum

In [ ]:
# Convert displacement_cum Batch to standard DataArray
sub_disp = displacement_cum.isel(date=slice(1, None)).to_dataset().VV.compute()

# We want 4x4 subplots for all 15 dates
fig, axes = plt.subplots(4, 4, figsize=(14, 12))
vmin = sub_disp.quantile(0.001).item()
vmax = sub_disp.quantile(0.999).item()

for i, ax in enumerate(axes.flat):
    if i >= sub_disp.sizes["date"]:
        ax.axis("off")
        continue
    d = sub_disp.isel(date=i)
    im = ax.imshow(d.values, cmap='RdBu_r', vmin=vmin, vmax=vmax)
    
    date_str = str(d.date.values)[:10]
    ax.set_title(date_str, fontsize=9)
    ax.axis("off")

# Hide any remaining unused subplots
for j in range(sub_disp.sizes["date"], len(axes)):
    axes[j].axis("off")

fig.colorbar(im, ax=axes, shrink=0.6, label='Displacement [m]')
plt.show()


In [ ]:
#displacement_cum.isel(date=slice(1,None)).sel(y=slice(1_266_000, 1_269_000), x=slice(622_000, 625_000)).plot(quantile=[0.001, 0.999], alpha=0.8)
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['axes.facecolor'] = 'white'
plt.rcParams['savefig.facecolor'] = 'white'

displacement_cum.isel(date=slice(1, None)).sel(
    y=slice(1_266_000, 1_269_000), x=slice(622_000, 625_000)
).plot(quantile=[0.001, 0.999], alpha=0.8)


In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(
    sub.isel(date=0).values,
    cmap='RdBu_r',
    vmin=sub.quantile(0.001).item(),
    vmax=sub.quantile(0.999).item(),
)
plt.colorbar(im, ax=ax)
ax.set_title(str(sub.date.values[0])[:10])
plt.show()

In [ ]:
n = 20
fig, axes = plt.subplots(3, 3, figsize=(12, 8))
for i, ax in enumerate(axes.flat):
    if i >= sub.sizes["date"]:
        ax.axis("off")
        continue
    d = sub.isel(date=i)
    im = ax.imshow(d.values, cmap='RdBu_r', vmin=-0.015, vmax=0.015)
    ax.set_title(str(sub.date.values[i])[:10], fontsize=9)
    ax.axis("off")
fig.colorbar(im, ax=axes, shrink=0.6)
plt.show()

In [ ]:
print("Number of dates:", displacement_cum.sizes["date"])
print(displacement_cum.date.values)

In [ ]:
sub = (
    displacement_cum
    .isel(date=slice(1, None))
    .sel(y=slice(1_266_000, 1_269_000), x=slice(622_000, 625_000))
    .to_dataset()
    .VV
    .compute()
)

print("dtype:", sub.dtype)
print("total NaN %:", float(sub.isnull().mean()) * 100)

if float(sub.isnull().mean()) < 1.0:
    print("valid min/max:", float(sub.min(skipna=True)), float(sub.max(skipna=True)))
else:
    print("valid min/max: N/A — 100% NaN in this window")

print("\nper-date NaN %:")
for i in range(sub.sizes["date"]):
    d = sub.isel(date=i)
    frac = float(d.isnull().mean())
    print(str(sub.date.values[i])[:10], f"{frac*100:.1f}% NaN")

## Plot Pixel Displacement Time Series

In [ ]:
# define point coordinates (as in the PyGMTSAR notebook): lat = 32.43, lon = -115.15
y = 1_267_750
x = 623_900
# find nearest pixel to the defined coordinates
displacement_pixel = displacement_cum.to_dataset().VV.sel(y=y, x=x, method='nearest').fillna(0)
displacement_pixel


In [ ]:
(1000*displacement_pixel).plot.scatter('date')
(1000*displacement_pixel).plot(lw=0.5, color='w')
plt.title(f'Cumulative LOS Displacement\ny={y}, x={x}', fontsize=18)
plt.xlabel('Date', fontsize=14)
plt.ylabel('Displacement, [mm]', fontsize=16)
plt.grid()
plt.savefig('Cumulative LOS Displacement POI, [mm].jpg')

## 2D Interactive Map

The map is also available as a standalone web page, which can be saved and used locally or shared on any web platform: https://insar.dev/ui/Imperial_Valley_2015.html

In [ ]:
velocity, disp0 = displacement_cum.velocity()

In [ ]:
# convert velocity in mm/year to GeoJSON for mapping on ipyleaflet
geojson = (1000*velocity).sel(y=slice(1_260_000, 1_270_000), x=slice(620_000, 630_000)).downsample(5).compute().to_geojson(crs='EPSG:4326')


In [ ]:
# Parse the string FIRST
if geojson is None:
    raise ValueError(
        "to_geojson() returned None — the selected region may have no valid data. "
        f"Data y range: {float(velocity.to_dataset().y.min()):.0f} to {float(velocity.to_dataset().y.max()):.0f}, "
        f"x range: {float(velocity.to_dataset().x.min()):.0f} to {float(velocity.to_dataset().x.max()):.0f}"
    )
geojson = json.loads(geojson)
# Create a colormap for velocities
colormap = plt.get_cmap('turbo')
def velocity_to_color(velocity, limits=[-60, 60]):
    normalized = np.clip((velocity - limits[0]) / (limits[1] - limits[0]), 0, 1)
    return matplotlib.colors.to_hex(colormap(normalized))
# Embed styles directly into GeoJSON properties
for feature in geojson['features']:
    color = velocity_to_color(feature['properties']['VV'])
    feature['properties']['style'] = {
        'color': color,
        'weight': 1,
        'fillColor': color,
        'fillOpacity': 0.4
    }
# Initialize the map with Esri Satellite as default
location = [11.466, 76.136]
osm_layer = basemap_to_tiles(basemaps.OpenStreetMap.Mapnik)
osm_layer.base = True
osm_layer.name = 'OpenStreetMap'
esri_base = basemap_to_tiles(basemaps.Esri.WorldImagery)
esri_base.base = True
esri_base.name = 'Esri Satellite'
m = Map(center=location, zoom=12, layers=[osm_layer, esri_base], scroll_wheel_zoom=True, layout=dict(height='620px'))
# Add GeoJSON layer - already a dict now
geo_json = GeoJSON(
    data=geojson,
    style_callback=lambda feature: feature['properties']['style'],
    hover_style={'fillOpacity': 1},
    name='InSAR'
)
m.add_layer(geo_json)
m.add_control(LayersControl(position='topright'))
display(m)


## Save the Results

Save the results in geospatial data formats like to NetCDF, GeoTIFF and others. The both formats (NetCDF and GeoTIFF) can be opened in QGIS and other GIS applications.

In [ ]:
# save the accumulated displacement in mm as a GeoTIFF file
(1000*displacement_cum).to_dataset().VV[-1].rio.to_raster('displacement.tif')

In [ ]:
# save the interactive map as an HMTL file
m.save('Imperial_Valley_2015_ipyleaflet.html', title='InSAR Velocity Map')

## Export from Google Colab

In [ ]:
if 'google.colab' in sys.modules:
    from google.colab import files
    files.download('Imperial_Valley_2015_ipyleaflet.html')
    files.download('displacement.tif')